# Candidate Generator v2.5 — Prospective score-blind selection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-/blob/main/notebooks/candidate_generator_v2_5_prospective_selection_colab.ipynb)

Bu notebook mevcut Candidate Generator v2.3 artifact'larından **en fazla 256 yeni score-blind audit çifti** seçer.

- İşlem türü: full-index prospective audit selection
- Girdi: Drive'daki 1.107 hash-kayıtlı retrieval artifact'ı
- Çıktı: private selection CSV, skorsuz/boş review CSV ve integrity raporu
- Yapılmayanlar: görüntü açma/render, duplicate kararı, scorer, family, split veya model eğitimi
- Dataset ZIP'i gerekli değildir.

Önce **Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU** seç. Ardından **Çalışma zamanı > Tümünü çalıştır** de. İlk hücre Google Drive izni isteyecek. Private CSV'yi açma; notebook sonunda oluşan ZIP'i Codex'e gönder.

In [ ]:
# 1/4 — Drive, repo ve T4 ortamı.
from google.colab import drive
from pathlib import Path
import csv, hashlib, json, shutil, subprocess, sys, torch

drive.mount('/content/drive')

REPOSITORY_URL = 'https://github.com/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-.git'
PROJECT_ROOT = Path('/content/NNGA')
REQUIRED_COMMIT = '3064cdf7acd99d5c7bf333752e97079f5b87d99d'

if not (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', 'main', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)

CURRENT_COMMIT = subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True
).strip()
ancestor = subprocess.run(
    ['git', '-C', str(PROJECT_ROOT), 'merge-base', '--is-ancestor', REQUIRED_COMMIT, CURRENT_COMMIT]
)
assert ancestor.returncode == 0, f'Gerekli frozen commit yok: {REQUIRED_COMMIT}'
assert torch.cuda.is_available(), 'GPU seçilmemiş. T4 GPU runtime açıp tekrar çalıştır.'
print('Drive bağlandı.')
print('Git commit:', CURRENT_COMMIT)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2/4 — Frozen sözleşme ve Drive artifact ön kontrolü.
OUTPUT_DIR = Path('/content/drive/MyDrive/NNGA_OUTPUTS/candidate_generator_v2_3_full_retrieval_v1')
SELECTION_DIR = Path('/content/drive/MyDrive/NNGA_OUTPUTS/candidate_generator_v2_5_prospective_selection_v1')
CONFIG_PATH = PROJECT_ROOT / 'config' / 'candidate_generator_v2_5_prospective_audit.json'
BUILDER_PATH = PROJECT_ROOT / 'src' / 'build_candidate_generator_v2_5_prospective_audit.py'
EXPECTED_CONFIG_SHA256 = '2c4213d34c70991550b85ec2d4699a0c7cf49b804904cc9bb3b4d7518d8fe745'
EXPECTED_BUILDER_SHA256 = '865342f0eab1730b7bd0042b4bc9946784f3e1bfc75f591fcf2a72108c160c59'

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

required = {
    'v2.3 output': OUTPUT_DIR,
    'stage manifest': OUTPUT_DIR / 'stage_manifest.json',
    'run contract': OUTPUT_DIR / 'run_contract.json',
    'v2.5 config': CONFIG_PATH,
    'v2.5 builder': BUILDER_PATH,
}
for label, path in required.items():
    exists = path.is_dir() if label == 'v2.3 output' else path.is_file()
    print(f'{label}: {exists} — {path}')
    assert exists, f'Eksik: {label} — {path}'

assert sha256(CONFIG_PATH) == EXPECTED_CONFIG_SHA256, 'Frozen config hash uyuşmuyor.'
assert sha256(BUILDER_PATH) == EXPECTED_BUILDER_SHA256, 'Frozen builder hash uyuşmuyor.'
manifest = json.loads((OUTPUT_DIR / 'stage_manifest.json').read_text(encoding='utf-8'))
assert len(manifest['artifacts']) == 1107, f"1.107 artifact bekleniyordu: {len(manifest['artifacts'])}"

EXISTING_COMPLETE = False
if SELECTION_DIR.exists():
    report_path = SELECTION_DIR / 'selection_report.json'
    if report_path.is_file():
        existing = json.loads(report_path.read_text(encoding='utf-8'))
        EXISTING_COMPLETE = existing.get('status') == 'v2_5_prospective_selection_frozen_images_not_opened'
    assert EXISTING_COMPLETE, f'Tamamlanmamış selection klasörü var; silmeden Codex’e bildir: {SELECTION_DIR}'

print('Kayıtlı artifact: 1107')
print('Config ve builder hash: doğrulandı')
print('Tamamlanmış selection zaten var mı:', EXISTING_COMPLETE)
print('ZIP veya dataset klasörü gerekmiyor.')

In [ ]:
# 3/4 — Full score-blind selection; çıktı aynı anda Drive loguna yazılır.
from datetime import datetime

if EXISTING_COMPLETE:
    print('Tamamlanmış selection bulundu; pahalı işlem yeniden çalıştırılmadı.')
else:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    LOG_PATH = SELECTION_DIR.parent / f'candidate_generator_v2_5_selection_{timestamp}.log'
    command = [
        sys.executable, '-u', str(BUILDER_PATH),
        '--output-dir', str(OUTPUT_DIR),
        '--selection-dir', str(SELECTION_DIR),
        '--config', str(CONFIG_PATH),
        '--block-size', '256',
        '--phash-block-size', '128',
        '--device', 'cuda',
    ]
    print('V2.5 score-blind selection başlıyor.')
    print('Komut:', ' '.join(command))
    print('Log:', LOG_PATH)
    print('Uzun süre yalnız group/rank ilerleme satırları görünmesi normaldir.')
    with LOG_PATH.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        RETURN_CODE = process.wait()
    print('RETURN CODE:', RETURN_CODE)
    assert RETURN_CODE == 0, f'Selection tamamlanamadı; Drive logunu Codex’e gönder: {LOG_PATH}'

assert (SELECTION_DIR / 'selection_report.json').is_file(), 'Selection raporu bulunamadı.'

In [ ]:
# 4/4 — Private içeriği göstermeden hash/blank-review kontrolü ve ZIP indirme.
from google.colab import files

REPORT_PATH = SELECTION_DIR / 'selection_report.json'
PRIVATE_PATH = SELECTION_DIR / 'selection_private.csv'
REVIEW_PATH = SELECTION_DIR / 'review_manifest.csv'
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
assert report['status'] == 'v2_5_prospective_selection_frozen_images_not_opened'
assert report['does_not_open_images'] is True
assert report['candidate_policy']['v2_5_full_pairs'] == 2994870
assert report['candidate_policy']['v2_5_pairs_per_image'] <= 25.0
assert report['input']['excluded_paths'] == 22922
assert report['input']['eligible_images'] == 107078
assert PRIVATE_PATH.is_file() and REVIEW_PATH.is_file()
assert sha256(PRIVATE_PATH) == report['integrity']['selection_private_sha256']
assert sha256(REVIEW_PATH) == report['integrity']['review_manifest_sha256']

with REVIEW_PATH.open('r', encoding='utf-8-sig', newline='') as handle:
    review_rows = list(csv.DictReader(handle))
assert len(review_rows) == report['selection']['selected_pairs']
assert len({row['pair_id'] for row in review_rows}) == len(review_rows)
assert all(not row['visual_verdict'] and not row['confidence'] for row in review_rows)
sensitive = {'selection_stratum', 'v2_5_candidate', 'robust_best_rank', 'phash_distance'}
assert sensitive.isdisjoint(review_rows[0].keys()) if review_rows else True

print('Durum:', report['status'])
print('Doğrulanan artifact:', report['input']['verified_artifacts'])
print('Dışlanan yol:', report['input']['excluded_paths'])
print('Eligible görüntü:', report['input']['eligible_images'])
print('Seçilen çift:', report['selection']['selected_pairs'])
print('Benzersiz görüntü:', report['selection']['unique_images'])
print('Boş review kararı:', report['selection']['blank_review_verdicts'])
print('Stratum hedefleri:', report['selection']['target_by_stratum'])
print('Stratum gerçek sayıları:', report['selection']['actual_by_stratum'])
print('Stratum eksikleri:', report['selection']['shortfall_by_stratum'])
print('V2.5 candidate:', report['candidate_policy']['v2_5_full_pairs'])
print('Çift/görüntü:', report['candidate_policy']['v2_5_pairs_per_image'])

PACKAGE_BASE = SELECTION_DIR.parent / 'candidate_generator_v2_5_prospective_selection_v1_package'
PACKAGE_PATH = Path(shutil.make_archive(str(PACKAGE_BASE), 'zip', root_dir=SELECTION_DIR))
print('Paket:', PACKAGE_PATH)
print('Private CSV’yi açma. Bu ZIP dosyasını doğrudan Codex’e gönder.')
files.download(str(PACKAGE_PATH))